# Structured Information Extraction with Pydantic


In [ ]:
def TODO(todo: str = "Fill the blank"):
    raise ValueError(todo)

In [ ]:
import os
import json
import pydantic
from openai import OpenAI
from PIL import Image

from enum import Enum
from typing import Optional
from datetime import date
from pydantic import BaseModel, Field
from pydantic import ValidationError
from pydantic import field_validator, model_validator

from ie_course.image import encode_image


In [ ]:
img_bytes = encode_image("../../data/gemini_generated_invoice.png")

key = TODO("Environment variable containing the API key")
model = TODO("Model name")
client = OpenAI(api_key=key, base_url="https://inference.dev.ellipsis-drive.com/v1")
model = "qwen3.5-122b-a10b"

# Part 1: Pydantic + Json-schema-based Extraction

## Define a schema with Pydantic

Instead of describing the shape in prose, we declare it once as a Pydantic model. This single
artifact is the **schema** we send to the server, the **validator** we run on the reply, and the
**typed object** we ultimately work with.

Note the things doing real work:
- `Field(description=...)` — sent to the model as part of the schema. Free prompting.
- `Enum` — constrains the *value*, not just the type.
- a nested model (`LineItem`) inside a `list` — repeated entities.
- `Optional[...] = None` — strict schemas have no "missing"; optional means "may be null".

In [ ]:
class InvoiceStatus(str, Enum):
    paid = "paid"
    unpaid = "unpaid"
    partially_paid = "partially_paid"

class LineItem(BaseModel):
    description: str = Field(description="Name/description of the purchased item")
    TODO("Add a field for the quantity and constrain it, such that it cannot be negative")
    unit_price: float = Field(description="Price per single unit, in the invoice currency")

class Invoice(BaseModel):
    vendor_name: str = Field(description="The company that issued the invoice")
    invoice_number: str = Field(description="The invoice identifier, e.g. INV-2024-0915", examples=TODO("Instead of using examples in the descriptions, put them in the example field"))
    issue_date: date = Field(description="Date the invoice was issued (ISO 8601)")
    currency: str = Field(description="3-letter ISO currency code, e.g. USD", examples=TODO("Instead of using examples in the descriptions, put them in the example field"))
    status: InvoiceStatus = TODO("This information is missing on the document. Make it optional to avoid hallucination")
    line_items: list[LineItem] = Field(description="The individual billed items")
    total_amount: float = Field(description="The total amount due")


print("Schema defined. Fields:", list(Invoice.model_fields))

## Inspect the schema the server and constrained decoding engine receive


In [ ]:
schema = TODO("Create the JSON schema from the Invoice model")
print(json.dumps(schema, indent=2))

In [ ]:
def chat_json(messages, schema, max_tokens=16_000):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "extraction", "schema": schema, "strict": True},
        },
        TODO("Make model output deterministic")
    )
    return TODO("Return the raw response string")

In [ ]:
messages = TODO("Define the messages for the extraction request")
raw = chat_json(messages, schema)
data = json.loads(raw)
invoice = Invoice.model_validate(data)

In [ ]:
print(invoice)

# Part 2: Pydantic Validation

#### Compute the line-item sum and compare it with invoice.total_amount.

In [ ]:
computed = TODO("Sum over quantity * unit_price for all line items")
print("Computed:", computed)
print("Invoice total:", invoice.subtotal)

#### Add a model_validator which checks if subtotal == sum over quantity * unit_price for all line items

In [ ]:
class ValidatedInvoice(Invoice):
    @field_validator("total_amount")
    @classmethod
    def total_must_be_positive(cls, v):
        if v < 0:
            raise ValueError("total_amount must be positive")
        return v

    @field_validator("currency")
    @classmethod
    def currency_is_three_letters(cls, v):
        if len(v) != 3 or not v.isalpha():
            raise ValueError("currency must be a 3-letter ISO code")
        return v.upper()

    @model_validator(mode="after")
    def check_sum(self):
        computed = sum([x.quantity * x.unit_price for x in self.line_items])
        if not np.isclose(computed, self.subtotal):
            raise ValueError("Model subtotal and sum over quantity * unit_price for all line items dont match.")
        return self
    

# Demonstrate the validator firing on bad data:
try:
    ValidatedInvoice.model_validate(invoice.model_dump())
except ValidationError as e:
    print("Validation failed:\n", e)

### Retry Loop

Lets change the ```field_validator("total_amount")``` check to raise an error if the total_amount field is positive - raising an error which states this field needs to be negative. 

Then, in a retry loop, validate the model output and in case of an error, add the error message to the chat history, asking the model to correct its own output. 

**Note** We should definitely use a **small** number of max_attempts here in order to not get stuck in long loops where the model gets confused and the chat history grows rapidly.

In [ ]:
class ValidatedInvoice(Invoice):
    @field_validator("total_amount")
    @classmethod
    def total_must_be_negative(cls, v):
        if TODO("Lets assume the model has to return negative values for this field"):
            raise ValueError("total_amount must be negative")
        return v


def extract_with_retries(img_bytes, model_cls=ValidatedInvoice, max_attempts=3):
    schema = model_cls.model_json_schema()
    messages = [
        {"role": "system", "content": "Extract the invoice as one JSON object matching "
                                       "the schema."},
        {"role": "user", "content": [
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{img_bytes}"}},
        ]},
    ]

    last_error = None
    for attempt in range(1, max_attempts + 1):
        raw = chat_json(messages, schema)
        try:
            return model_cls.model_validate_json(raw)
        except ValidationError as e:
            last_error = e
            print(f"Attempt {attempt} failed validation; feeding the error back.")
            # Keep the bad answer in context, then add a targeted correction turn.
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": TODO(
                "Write a correction prompt that quotes str(e) and asks the model to return "
                "the full corrected JSON object")})
            
    raise last_error

In [ ]:
def extract_via_tool(img_bytes):

    tools = [{
        "type": "function",
        "function": {
            "name": "save_invoice",
            "description": "Save the structured invoice extracted from the text.",
            "parameters": TODO("paste the schema"),
        },
    }]

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Extract the invoice by calling save_invoice."},
            {
                "role": "user", 
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_bytes}"}
                    },
                ]
            }
        ],
        tools=tools,
        tool_choice={"type": "function", "function": {"name": "save_invoice"}},
    )
    
    TODO("return the function arguments")


tool_params = extract_via_tool(img_bytes)
tool_invoice = Invoice.model_validate_json(tool_invoice)

In [ ]:
print(tool_params)

### Validators catch the *sporadic* semantic errors

Strict schemas guarantee types, not meaning. A `field_validator` is the right place to catch
errors the model makes occasionally — e.g. a negative total. **Caveat from the lecture:** if the
model gets something wrong *consistently*, retries fail consistently too.

In [ ]:
class ValidatedInvoice(Invoice):
    @field_validator("total_amount")
    @classmethod
    def total_must_be_positive(cls, v):
        if v < 0:
            raise ValueError("total_amount must be positive")
        return v

    @field_validator("currency")
    @classmethod
    def currency_is_three_letters(cls, v):
        if len(v) != 3 or not v.isalpha():
            raise ValueError("currency must be a 3-letter ISO code")
        return v.upper()

    @model_validator()
    @classmethod
    def check_sum(self):
        # use np.close to avoid rounding errors on byte level
        return np.isclose(sum([item.quantity * item.unit_price for item in self.line_items]), self.total_amount) 
    

# Demonstrate the validator firing on bad data:
try:
    ValidatedInvoice.model_validate(invoice.model_dump())
except ValidationError as e:
    print("Validation failed:\n", e)

In [ ]:
import numpy as np